In [ ]:
# 자전거 대여량 예측(mlflow 사용)
# 사전설치 : pip install mlflow
# MLflow UI 실행 : 터미널에서 mlflow ui  ---> http://localhost:5000에 접속하여 실험결과, 파라미터, metrics, 저장된 모델  확인
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
# === MLflow 임포트 ===
import mlflow
import mlflow.keras
import mlflow.tensorflow

In [ ]:
# 폰트지정
plt.rcParams['font.family'] = 'Malgun Gothic'

In [ ]:
# 1. 데이터 불러오기
dataset_path = './dataset/'
train_data = pd.read_csv(dataset_path + 'train.csv')
test_data = pd.read_csv(dataset_path + 'test.csv')

In [ ]:
# 2. 데이터 전처리
def preprocess_data(data):
    data['datetime'] = pd.to_datetime(data['datetime'])
    data['hour'] = data['datetime'].dt.hour
    data['day'] = data['datetime'].dt.day
    data['month'] = data['datetime'].dt.month
    data['year'] = data['datetime'].dt.year

    data = data.drop(['datetime', 'casual', 'registered'], axis=1, errors='ignore')
    return data

train_data = preprocess_data(train_data)
test_data = preprocess_data(test_data)

# Feature와 Target 분리
X = train_data.drop(['count'], axis=1).values
y = train_data['count'].values.reshape(-1, 1)

# MinMaxScaler를 사용하여 데이터 정규화
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X = scaler_X.fit_transform(X)
y = scaler_y.fit_transform(y)

# Train/Test 데이터 분리
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 3. 데이터 차원 조정 (LSTM, GRU 입력 형태로 변환)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_val = np.reshape(X_val, (X_val.shape[0], X_val.shape[1], 1))

# === MLflow 실험 설정 ===
mlflow.set_experiment("Bike_Sharing_Prediction_Comparison")
# === 비교할 모델 리스트 정의 ===
models_to_train = ['GRU', 'LSTM']
histories = {} # 시각화 비교를 위해 학습 기록 저장용 딕셔너리

for model_type in models_to_train:
    print(f"\n========== {model_type} 모델 학습 시작 ==========")

    # 각 모델별로 MLflow run 실행
    with mlflow.start_run(run_name=f"{model_type}_model_training"):
        # 파라미터 로깅
        epochs = 20
        batch_size = 16
        units_1 = 64
        units_2 = 32
        dropout = 0.2

        mlflow.log_param("model_type", model_type) # [수정] 모델 타입 로깅
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("units_1", units_1)
        mlflow.log_param("units_2", units_2)
        mlflow.log_param("dropout_rate", dropout)

        # 4. 모델 생성 (조건문으로 GRU/LSTM 분기)
        model = Sequential()
        if model_type == 'GRU':
            model.add(GRU(units_1, return_sequences=True, input_shape=(X_train.shape[1], 1)))
            model.add(Dropout(dropout))
            model.add(GRU(units_2))
            model.add(Dropout(dropout))
        elif model_type == 'LSTM':
            model.add(LSTM(units_1, return_sequences=True, input_shape=(X_train.shape[1], 1)))
            model.add(Dropout(dropout))
            model.add(LSTM(units_2))
            model.add(Dropout(dropout))

        model.add(Dense(1))

        # 5. 모델 컴파일 및 학습
        model.compile(optimizer='adam', loss='mse')
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            verbose=1,
            shuffle=False
        )

        # 히스토리 저장 (나중에 그래프 그리기 위함)
        histories[model_type] = history

        # === MLflow 메트릭 및 모델 로깅 ===
        # 학습 손실 및 검증 손실 로깅
        for epoch, loss in enumerate(history.history['loss']):
            mlflow.log_metric("train_loss", loss, step=epoch)
        for epoch, val_loss in enumerate(history.history['val_loss']):
            mlflow.log_metric("val_loss", val_loss, step=epoch)

        # 모델 저장
        mlflow.keras.log_model(model, f"{model_type}_model")

        # 7. 테스트 데이터 예측 및 결과 개별 저장
        y_pred = model.predict(X_test)
        y_pred = scaler_y.inverse_transform(y_pred)

        # 결과 저장 (모델별로 파일명 다르게 저장)
        output_filename = f'test_predictions_{model_type}.csv'
        test_data_copy = test_data.copy()
        test_data_copy['predicted_count'] = y_pred
        test_data_copy.to_csv(dataset_path + output_filename, index=False)
        print(f"{model_type} 예측 결과가 '{output_filename}'로 저장되었습니다.")

In [ ]:
# 6. 학습 과정 비교 시각화 (두 모델 한 번에 비교)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for model_type, history in histories.items():
    plt.plot(history.history['loss'], label=f'{model_type} Train Loss')
plt.title('모델별 학습 손실 비교')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.subplot(1, 2, 2)
for model_type, history in histories.items():
    plt.plot(history.history['val_loss'], label=f'{model_type} Val Loss', linestyle='--')
plt.title('모델별 검증 손실 비교')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.tight_layout()
plt.show()